In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from scipy import stats

In [27]:
reviews_df = pd.read_csv("final_yelp_dataset.csv")
reviews_df = reviews_df[reviews_df['tier'] != '$$$$']
reviews_df.head(5)

reviews_sample = reviews_df.sample(n = 10000, random_state = 42)

In [28]:
roberta_model = pipeline(
    "sentiment-analysis",
    model = "cardiffnlp/twitter-roberta-base-sentiment",
    device = 0
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 35065.93it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
roberta_results = roberta_model(
    reviews_sample['text'].tolist(),
    truncation=True,
    max_length=128,
    batch_size=128
)

In [34]:
reviews_sample['roberta_label'] = [r['label'] for r in roberta_results]
reviews_sample['roberta_score'] = [r['score'] for r in roberta_results]

In [35]:
def convert_score(r):
    if r['label'] == 'LABEL_2':
        return r['score']
    elif r['label'] == 'LABEL_1':
        return 0.5
    else:
        return 1 - r['score']

reviews_sample['roberta_sentiment'] = [
    convert_score(r) for r in roberta_results
]


In [45]:
cuisine_scores = (
    reviews_sample
    .groupby('cuisine')['roberta_sentiment']
    .mean()
    .sort_values()
)

print("Mean sentiment by cuisine:")
print(cuisine_scores)

Mean sentiment by cuisine:
cuisine
American    0.734170
Chinese     0.743792
Mexican     0.748817
Korean      0.772869
Japanese    0.776083
Greek       0.777938
Italian     0.782916
Indian      0.791432
French      0.816892
Thai        0.825350
Name: roberta_sentiment, dtype: float64


In [46]:
groups = [
    group['roberta_sentiment'].values
    for _, group in reviews_sample.groupby('cuisine')
]

f_stat, p_val = stats.f_oneway(*groups)

print(f"ANOVA: F = {f_stat:.2f}, p = {p_val:.5f}")


ANOVA: F = 5.20, p = 0.00000


In [42]:
american = reviews_sample[
    reviews_sample['cuisine'] == 'American'
]['roberta_sentiment']

print("Comparisons vs American:")

for cuisine in reviews_sample['cuisine'].unique():
    if cuisine == 'American':
        continue
        
    other = reviews_sample[
        reviews_sample['cuisine'] == cuisine
    ]['roberta_sentiment']
    
    t_stat, p_val = stats.ttest_ind(american, other)
    diff = other.mean() - american.mean()
    
    print(f"{cuisine}: diff = {diff:.4f}, p = {p_val:.5f}")

Comparisons vs American:
Chinese: diff = 0.0096, p = 0.52453
Mexican: diff = 0.0146, p = 0.19116
Japanese: diff = 0.0419, p = 0.00135
Indian: diff = 0.0573, p = 0.02996
Italian: diff = 0.0487, p = 0.00001
Thai: diff = 0.0912, p = 0.00012
French: diff = 0.0827, p = 0.01664
Korean: diff = 0.0387, p = 0.29531
Greek: diff = 0.0438, p = 0.14705


In [44]:
corr, _ = stats.pearsonr(
    reviews_sample['roberta_sentiment'],
    reviews_sample['individual_stars']
)

print(f"Pearson r = {corr:.4f}")

tier_scores = (
    reviews_sample
    .groupby(['tier', 'cuisine'])['roberta_sentiment']
    .mean()
    .reset_index()
)

print("Sentiment by tier:")
print(tier_scores)

american_tiers = (
    reviews_sample[reviews_sample['cuisine'] == 'American']
    .groupby('tier')['roberta_sentiment']
    .mean()
)

print("American by tier:")
print(american_tiers)

print("Spread:")
print(cuisine_scores.max() - cuisine_scores.min())

Pearson r = 0.8687
Sentiment by tier:
   tier   cuisine  roberta_sentiment
0     $  American           0.568631
1     $   Chinese           0.685210
2     $    French           0.696104
3     $     Greek           0.746405
4     $    Indian           0.674493
5     $   Italian           0.687593
6     $  Japanese           0.905873
7     $    Korean           0.796261
8     $   Mexican           0.747175
9     $      Thai           0.687870
10   $$  American           0.757424
11   $$   Chinese           0.772665
12   $$    French           0.856306
13   $$     Greek           0.793608
14   $$    Indian           0.798352
15   $$   Italian           0.795885
16   $$  Japanese           0.770781
17   $$    Korean           0.767906
18   $$   Mexican           0.747850
19   $$      Thai           0.828963
20  $$$  American           0.795804
21  $$$   Chinese           0.857205
22  $$$    French           0.757091
23  $$$     Greek           0.763883
24  $$$   Italian           0.789098
